## Creating input-target Pairs
#### This is the last step before vector embbeding.

In [12]:
import tiktoken

with open("Data/verdict.txt","r",encoding="utf-8") as f:
    verdict = f.read()

tokenizer = tiktoken.get_encoding("gpt2")

enc_text = tokenizer.encode(verdict)
print(len(enc_text))

5774


In [13]:
enc_sample=enc_text[50:]

In [14]:
context_size=4
x=enc_sample[:context_size]
y=enc_sample[1:context_size+1]
print(f"x : {x}")
print(f"y : {y}")

x : [11, 339, 550, 5710]
y : [339, 550, 5710, 465]


In [15]:
for i in range(1,context_size+1):
    x=enc_sample[:i]
    y=enc_sample[i]
    print(f"Input:{x} ---> Target:{y}")

Input:[11] ---> Target:339
Input:[11, 339] ---> Target:550
Input:[11, 339, 550] ---> Target:5710
Input:[11, 339, 550, 5710] ---> Target:465


In [16]:
for i in range(1,context_size+1):
    x=enc_sample[:i]
    y=enc_sample[i]
    print(f"Input: {tokenizer.decode(x)} ---> Target:{tokenizer.decode([y])}")

Input: , ---> Target: he
Input: , he ---> Target: had
Input: , he had ---> Target: dropped
Input: , he had dropped ---> Target: his


## Implementing a Data Loader 
#### For a efficient data-loader implementation, we use PyTorch's  in built Dataset and DataLoader Classes.


In [17]:
import tiktoken,torch
from torch.utils.data import TensorDataset,DataLoader
tokenizer = tiktoken.get_encoding("gpt2")
input_ids=[]
target_ids=[]
txt = open("Data/verdict.txt","r",encoding="utf-8").read()
stride=1
context_size=4
token_ids=tokenizer.encode(txt,allowed_special={"<|endoftext|>"})
for i in range(0,len(token_ids)-context_size,stride):
    input_chunck=token_ids[i:i+context_size]
    target_chunck=token_ids[i+1:i+context_size+1]
    input_ids.append(input_chunck)
    target_ids.append(target_chunck)

input_tensor = torch.tensor(input_ids, dtype=torch.long)
target_tensor = torch.tensor(target_ids, dtype=torch.long)

# Tensor shapes
print("Input Tensor Shape :", input_tensor.shape)
print("Target Tensor Shape:", target_tensor.shape)

print("\nFirst 5 Samples\n")

# Display first 5 samples
for i in range(min(5, len(input_tensor))):

    print(f"Sample {i}")

    print("Input Tokens : ", input_tensor[i].tolist())
    print("Target Tokens:", target_tensor[i].tolist())

    print(
        "Input Text   :",tokenizer.decode(input_tensor[i].tolist())
    )

    print(
        "Target Text  :",tokenizer.decode(target_tensor[i].tolist())
    )

    print("-" * 60)



Input Tensor Shape : torch.Size([5770, 4])
Target Tensor Shape: torch.Size([5770, 4])

First 5 Samples

Sample 0
Input Tokens :  [10970, 33310, 35, 18379]
Target Tokens: [33310, 35, 18379, 198]
Input Text   : THE VERDICT
Target Text  :  VERDICT

------------------------------------------------------------
Sample 1
Input Tokens :  [33310, 35, 18379, 198]
Target Tokens: [35, 18379, 198, 15749]
Input Text   :  VERDICT

Target Text  : DICT
June
------------------------------------------------------------
Sample 2
Input Tokens :  [35, 18379, 198, 15749]
Target Tokens: [18379, 198, 15749, 40417]
Input Text   : DICT
June
Target Text  : ICT
June 1908
------------------------------------------------------------
Sample 3
Input Tokens :  [18379, 198, 15749, 40417]
Target Tokens: [198, 15749, 40417, 198]
Input Text   : ICT
June 1908
Target Text  : 
June 1908

------------------------------------------------------------
Sample 4
Input Tokens :  [198, 15749, 40417, 198]
Target Tokens: [15749, 40417,

In [18]:
dataset = TensorDataset(input_tensor, target_tensor)

loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True
)

for x, y in loader:
    print(x.shape)
    print(y.shape)
    break

torch.Size([8, 4])
torch.Size([8, 4])


# Creating Vector Embeddings

In [19]:
input_ids=torch.tensor([2,3,5,1])

In [20]:
vocabulary_size=6
output_dimensions=3

torch.manual_seed(123)
embedding_layer=torch.nn.Embedding(vocabulary_size,output_dimensions)

In [21]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [22]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [23]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


### Hands-on Experince on postinal embedding 

In [ ]:
import torch
vocabulary_size=50257
output_dimensions=768

embedding_layer=torch.nn.Embedding(vocabulary_size,output_dimensions)

In [36]:
from GPTDataSetV1 import create_dataloader_v1
max_length = 4

raw_txt = open("Data/verdict.txt","r",encoding="utf-8").read()

dataloader = create_dataloader_v1(
    txt=raw_txt,
    batch_size=8,
    context_size=max_length,
    stride=max_length,
    shuffle=False
)

data_iter = iter(dataloader)

inputs, targets = next(data_iter)




In [40]:
print("Input Shape",inputs.shape)
print("\nInputs: ",inputs)

Input Shape torch.Size([8, 4])

Inputs:  tensor([[10970, 33310,    35, 18379],
        [  198, 15749, 40417,   198],
        [  198,    40,   550,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  198,   198, 11274,  5891],
        [ 1576,   438,   568,   340]])


In [39]:
print("Target Shape",targets.shape)
print("\nTargets:",targets)

Target Shape torch.Size([8, 4])

Targets: tensor([[33310,    35, 18379,   198],
        [15749, 40417,   198,   198],
        [   40,   550,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   198],
        [  198, 11274,  5891,  1576],
        [  438,   568,   340,   373]])
